# 02. 상권 EDA와 PATH-v0 기준선

유동인구·점포·매출의 관계를 점검하고, 30분 접근성 후보 안에서 목적별 규칙 기반 순위 PATH-v0를 생성합니다. 이 노트북은 `01`의 시간대 공통 후보 결과가 있어야 합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/막시무스'  # 실제 경로로 수정
%cd $PROJECT_DIR
!pip -q install -r requirements-eda.txt
!pip -q install scikit-learn geopandas openpyxl

In [ ]:
from pathlib import Path
required = [
    'data/processed/local_transit_30min/time_period_candidates.csv',
    'data/raw/seoul_commercial_sales_dong_2025.csv',
    'data/raw/seoul_commercial_stores_dong_2025.csv',
    'data/raw/seoul_commercial_floating_population_dong.csv',
    'data/raw/seoul_attracting_facilities_dong.csv',
]
missing = [name for name in required if not Path(name).exists()]
for name in required:
    print(('OK  ' if name not in missing else '없음'), name)
if missing:
    raise FileNotFoundError('필수 파일을 data/raw 또는 data/processed에 추가하세요.')

## 1. 유동인구–매출 괴리 EDA

유동인구와 점포 수로 예측한 매출 대비 실제 매출의 반복 괴리를 확인합니다. 괴리는 인과효과가 아니라 후속 조사 후보를 고르는 지표입니다.

In [ ]:
!python -m scripts.flow_sales_gap
import pandas as pd
gap = pd.read_csv('data/processed/flow_sales_gap/flow_sales_gap_candidates.csv')
display(gap.head(20))
print(open('data/processed/flow_sales_gap/EDA_REPORT.md', encoding='utf-8').read()[:3000])

## 2. 목적별 특징 변수

식사·카페·공부·쇼핑·여가문화별로 점포 구성, 시설, 시간대 소비 신호를 분리합니다. 현재 행정동 PATH-v0는 B078 기반 학습모형의 투명한 비교 기준입니다.

In [ ]:
!python -m scripts.build_purpose_features
!python -m scripts.build_purpose_ranking_baseline
rankings = pd.read_csv('data/processed/purpose_rankings/path_v0_top10_by_purpose.csv')
for purpose, group in rankings.groupby('purpose'):
    print(f'\n[{purpose}]')
    display(group[['purpose_rank', 'dong_name', 'sggnm', 'path_v0_score', 'recommendation_reason']].head(5))

## 해석 주의

PATH-v0는 개별 사용자 취향·실시간 경로·실제 B078 목적 이동량을 예측하는 모델이 아닙니다. B078 전체 원본을 확보한 뒤 동일 출발지·시간대 조건에서 Conditional Logit, LightGBM, CatBoost와 Recall@5·NDCG@5·MRR로 비교합니다.